In [ ]:
import itertools
import random
from pathlib import Path

In [ ]:
!pip install networkx matplotlib


In [ ]:
import matplotlib.pyplot as plt

def draw_reversible_circuit(circuit, n_inputs, title="Reversible Circuit"):
    fig, ax = plt.subplots(figsize=(1.5 * len(circuit), 1.5 * n_inputs))
    ax.set_title(title)
    ax.axis('off')

    # Draw horizontal lines (wires)
    for i in range(n_inputs):
        ax.plot([0, len(circuit)], [i, i], color='black', linewidth=1)

    # Draw gates
    for col, gate in enumerate(circuit):
        gate_type, controls, target = gate
        x = col + 0.5

        # Draw controls as solid circles
        for c in controls:
            ax.plot(x, c, 'ko', markersize=8)

        # Draw target as a circle with a plus (Toffoli style)
        ax.plot(x, target, 'o', markersize=14, markeredgecolor='black', markerfacecolor='white')
        ax.plot([x, x], [target - 0.3, target + 0.3], color='black')
        ax.plot([x - 0.3, x + 0.3], [target, target], color='black')

    plt.ylim(-1, n_inputs)
    plt.xlim(0, len(circuit))
    plt.gca().invert_yaxis()
    plt.show()


In [ ]:
# ---------- Gate Simulation Logic ---------- #

def apply_toffoli(inputs, controls, target):
    if inputs[controls[0]] == 1 and inputs[controls[1]] == 1:
        inputs[target] ^= 1
    return inputs

def simulate_circuit(circuit, input_vector):
    vector = input_vector.copy()
    for gate in circuit:
        gate_type = gate[0]
        if gate_type == "TOFFOLI":
            controls, target = gate[1], gate[2]
            if all(vector[c] == 1 for c in controls):
                vector[target] ^= 1
    return vector

In [ ]:
# ---------- Fault List Generator (MGF) ---------- #

def generate_fault_list(circuit):
    fault_list = []
    for i in range(len(circuit)):
        faulty_circuit = circuit[:i] + circuit[i+1:]
        fault_list.append((i, faulty_circuit))
    return fault_list

In [ ]:
# ---------- Fault Detection and DP Caching ---------- #

def detect_fault(dp, vector, fault_id, circuit, faulty_circuit):
    key = (tuple(vector), fault_id)
    if key in dp:
        return dp[key]
    normal_output = simulate_circuit(circuit, vector)
    faulty_output = simulate_circuit(faulty_circuit, vector)
    detected = normal_output != faulty_output
    dp[key] = detected
    return detected

In [ ]:
# ---------- Fitness Evaluation ---------- #

def compute_fitness(individual, fault_list, circuit, dp, alpha=1.0, beta=0.1):
    detected_faults = set()
    for vector in individual:
        for fault_id, faulty_circuit in fault_list:
            if detect_fault(dp, vector, fault_id, circuit, faulty_circuit):
                detected_faults.add(fault_id)
    coverage = len(detected_faults) / len(fault_list)
    fitness = alpha * coverage - beta * len(individual)
    return fitness, coverage, detected_faults


In [ ]:
# ---------- GA Operators ---------- #

def crossover(parent1, parent2):
    cut1 = random.randint(1, len(parent1) - 1)
    cut2 = random.randint(1, len(parent2) - 1)
    return parent1[:cut1] + parent2[cut2:]

def mutate(individual, mutation_rate=0.2):
    for vec in individual:
        if random.random() < mutation_rate:
            bit = random.randint(0, len(vec) - 1)
            vec[bit] ^= 1
    return individual

In [ ]:
# ---------- GA Main Routine ---------- #

def run_ga(circuit, n_inputs, generations=30, pop_size=10):
    fault_list = generate_fault_list(circuit)
    dp = {}
    # Population initialization with variable-length individuals
    population = [
        [ [random.randint(0, 1) for _ in range(n_inputs)] for _ in range(random.randint(2, 8)) ]
        for _ in range(pop_size)
    ]

    for _ in range(generations):
        fitness_scores = [compute_fitness(ind, fault_list, circuit, dp) for ind in population]
        population = [x for _, x in sorted(zip(fitness_scores, population), reverse=True)]
        next_gen = population[:pop_size // 2]
        while len(next_gen) < pop_size:
            p1, p2 = random.sample(next_gen, 2)
            child = crossover(p1, p2)
            child = mutate(child)
            next_gen.append(child)
        population = next_gen
    best = max(population, key=lambda ind: compute_fitness(ind, fault_list, circuit, dp)[0])
    best_fitness, coverage, detected_faults = compute_fitness(best, fault_list, circuit, dp)
    return best, (best_fitness, coverage, len(fault_list), detected_faults)



In [ ]:
# ---------- File Handling ---------- #

def read_real_file(file_path):
    with open(file_path, 'r') as file:
        return file.readlines()

def parse_real_file_lines(real_lines):
    var_map = {}
    circuit = []
    for line in real_lines:
        if line.startswith(".variables"):
            vars_list = line.strip().split()[1:]
            var_map = {var: idx for idx, var in enumerate(vars_list)}
        elif line.startswith(".begin"):
            continue
        elif line.startswith(".end"):
            break
        elif line.startswith("t"):
            tokens = line.strip().split()
            wires = [var_map[x] for x in tokens[1:]]
            if len(wires) == 1:
                circuit.append(("TOFFOLI", [], wires[0]))
            else:
                circuit.append(("TOFFOLI", wires[:-1], wires[-1]))
    return circuit, len(var_map)

In [ ]:
# ---------- Main Runner for Folder ---------- #
import csv
def run_on_folder(folder_path, save_csv_path="ga_results.csv"):
    import csv
    results = []
    folder = Path(folder_path)
    real_files = list(folder.glob("*.real"))

    for file in real_files:
        try:
            lines = read_real_file(file)
            circuit, n_inputs = parse_real_file_lines(lines)

            best_test_set, (fitness, coverage, total_faults, detected_faults) = run_ga(circuit, n_inputs)

            results.append({
                "file": file.name,
                "fitness": round(fitness, 4),
                "coverage": round(coverage, 4),
                "total_faults": total_faults,
                "detected_faults": len(detected_faults),
                "detected_ids": list(sorted(detected_faults)),
                "test_vectors": best_test_set,
                "num_test_vectors": len(best_test_set),
                "num_gates": len(circuit)  # ✅ Now correctly added
            })

        except Exception as e:
            results.append({
                "file": file.name,
                "fitness": "ERROR",
                "coverage": "ERROR",
                "total_faults": "ERROR",
                "detected_faults": "ERROR",
                "detected_ids": None,
                "test_vectors": None,
                "num_test_vectors": None,
                "num_gates": None,  # ✅ Also added to error case
                "error": str(e)
            })

    # Save to CSV
    with open(save_csv_path, "w", newline='') as f:
        writer = csv.DictWriter(f, fieldnames=[
            "file", "fitness", "coverage", "total_faults", "detected_faults",
            "num_test_vectors", "num_gates"
        ])
        writer.writeheader()
        for row in results:
            if row["fitness"] != "ERROR":
                writer.writerow({key: row[key] for key in writer.fieldnames})

    return results


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_gates_coverage_vectors(results):
    # Filter valid results
    filtered = [r for r in results if r["fitness"] != "ERROR"]

    files = [r["file"] for r in filtered]
    num_gates = [r["num_gates"] for r in filtered]
    coverage = [r["coverage"] for r in filtered]
    num_vectors = [r["num_test_vectors"] for r in filtered]

    x = np.arange(len(files))  # X-axis positions

    fig, ax1 = plt.subplots(figsize=(12, 6))

    # Bar plot for number of gates
    bars = ax1.bar(x - 0.2, num_gates, width=0.4, color='lightcoral', label="Gate Count")
    ax1.set_ylabel("Number of Gates", color="lightcoral")
    ax1.tick_params(axis='y', labelcolor="lightcoral")
    ax1.set_xticks(x)
    ax1.set_xticklabels(files, rotation=45, ha='right')

    # Second axis for coverage
    ax2 = ax1.twinx()
    ax2.plot(x, coverage, color='royalblue', marker='o', label="Fault Coverage")
    ax2.set_ylabel("Coverage", color="royalblue")
    ax2.tick_params(axis='y', labelcolor="royalblue")
    ax2.set_ylim(0, 1.05)

    # Third axis for number of test vectors
    ax3 = ax1.twinx()
    ax3.spines['right'].set_position(('outward', 60))  # offset third y-axis
    ax3.plot(x, num_vectors, color='darkgreen', marker='s', linestyle='--', label="Test Vectors")
    ax3.set_ylabel("Number of Test Vectors", color="darkgreen")
    ax3.tick_params(axis='y', labelcolor="darkgreen")

    # Title and layout
    plt.title("Gates, Coverage & Test Vectors per Circuit")
    fig.tight_layout()

    # Combine legends from all axes
    lines, labels = [], []
    for ax in [ax1, ax2, ax3]:
        for line in ax.get_lines():
            lines.append(line)
            labels.append(line.get_label())
    fig.legend(lines + [bars], labels + ["Gate Count"], loc="upper right", bbox_to_anchor=(1.15, 1))

    plt.show()


In [ ]:
# ---------- Example Usage ---------- #

# Replace this path with your actual input folder path
input_folder_path = "/content/drive/MyDrive/Dissertation Work/Research Work/circuits for testing"
results = run_on_folder(input_folder_path, save_csv_path="ga_results.csv")
for r in results:
  print(r["file"], "→ gates:", r.get("num_gates"), "vectors:", r.get("num_test_vectors"))

fredkin_6.real → gates: 3 vectors: 2
3_17_13.real → gates: 6 vectors: 2
decod24-v0_38.real → gates: 6 vectors: 2
4gt4-v0_72.real → gates: 6 vectors: 2
4gt5_75.real → gates: 5 vectors: 2
4gt10-v1_81.real → gates: 6 vectors: 2
4mod7-v0_94.real → gates: 6 vectors: 2


In [ ]:
plot_coverage(result_table)

NameError: name 'plot_coverage' is not defined

[{'file': '3_17_13.real',
  'fitness': 0.1,
  'test_vectors': [[0, 0, 0], [0, 0, 0], [1, 1, 0]]},
 {'file': '4mod7-v0_94.real',
  'fitness': -0.2333,
  'test_vectors': [[1, 1, 1, 1, 1], [1, 0, 0, 1, 0], [1, 0, 1, 0, 0]]},
 {'file': '4gt5_75.real',
  'fitness': 0.1,
  'test_vectors': [[1, 0, 0, 1, 1], [1, 1, 1, 0, 1], [0, 1, 1, 1, 1]]},
 {'file': '4gt4-v0_72.real',
  'fitness': -0.4,
  'test_vectors': [[1, 1, 0, 1, 0], [1, 0, 0, 0, 1], [0, 1, 0, 0, 0]]},
 {'file': '4gt10-v1_81.real',
  'fitness': -0.4,
  'test_vectors': [[1, 0, 1, 1, 0], [1, 0, 1, 0, 1], [1, 0, 1, 1, 0]]},
 {'file': 'fredkin_6.real',
  'fitness': -0.2333,
  'test_vectors': [[1, 1, 0], [1, 1, 0], [0, 0, 0]]},
 {'file': 'decod24-v0_38.real',
  'fitness': 0.1,
  'test_vectors': [[1, 1, 0, 1], [0, 0, 1, 1], [0, 0, 1, 0]]}]